In [6]:
#export MODEL="Your model here
export MODEL="unsloth/Qwen3.6-35B-A3B-GGUF:UD-IQ4_NL"

common_memory_breakdown_print: | memory breakdown [MiB] | total   free     self   model   context   compute    unaccounted |
common_memory_breakdown_print: |   - CUDA0 (RTX 2060)   |  5737 =  545 + ( 4572 =  2781 +    1297 +     493) +         619 |
common_memory_breakdown_print: |   - Host               |                 15985 = 15853 +       0 +     131                |


In [25]:
llama-fit-params -hf $MODEL -ub 64 -ngl 99 -ncmoe 36 | tee args.txt # This is just for downloading and some initial values

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
common_download_file_single_online: HEAD failed, status: 404
no remote preset found, skipping
common_params_fit_impl: getting device memory data for initial parameters:
common_memory_breakdown_print: | memory breakdown [MiB] | total   free     self   model   context   compute       unaccounted |
common_memory_breakdown_print: |   - CUDA0 (RTX 2060)   |  5737 = 5075 + ( 8942 =  3509 +    5182 +     250) + 17592186036135 |
common_memory_breakdown_print: |   - Host               |                 13750 = 13685 +       0 +      65                   |
common_params_fit_impl: projected to use 8942 MiB of device memory vs. 5075 MiB of free device memory
common_params_fit_impl: cannot meet free memory target of 1024 MiB, need

In [45]:
# Configurable batch sizes to test to fit context
# Though it looks like batch sizes have no effects 
# as per README 
# "Increasing this value above the value of the physical batch size may improve prompt processing performance 
# when using multiple GPUs with pipeline parallelism. Default: `2048`" 

BATCH_SIZES="2048" # 2 args so we can see the diff, shoud not make a difference  
UBATCH_SIZES="128"
FITT="512"
CMOE="30 31 32 33 34 35 36 37 38 39 40"
CMOE="36"

echo "Testing different batch/ubatch/fitt/cmoe combinations for ${MODEL}..." 
  
for ub in $UBATCH_SIZES; do  
    for b in $BATCH_SIZES; do
        for ft in $FITT; do
            for cmoe in $CMOE; do
                # need to add fitt for nvidia
                OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub -fitt $ft -ncmoe $cmoe 2>&1)
                
                #echo "Raw output: $OUTPUT"  # Debug line  
                # Extract context and GPU layers more robustly  
                CTX=$(echo "$OUTPUT" | grep -o '\-c [0-9]*' | awk '{print $2}')  
                NGL=$(echo "$OUTPUT" | grep -o '\-ngl -\?[0-9]*' | awk '{print $2}')  
                MEM=$(echo "$OUTPUT" | awk '/stdout/{f=1;next} f&&NF{printf "%s:%s/%s/%s ",$1,$2,$3,$4}')

                echo -n "ub: $ub, b: $b, fitt: $ft, ncmoe: $cmoe. "
                if [ ! -z "$CTX" ] && [ ! -z "$NGL" ]; then  
                    echo "ctx: $CTX, ngl: $NGL"  
                else  
                    echo "No valid parameters found"  
                fi
            done
        done
    done  
done 

Testing different batch/ubatch/fitt/cmoe combinations for unsloth/Qwen3.6-35B-A3B-GGUF:UD-IQ4_NL...
ub: 128, b: 2048, fitt: 512, ncmoe: 36. ctx: 42240, ngl: -1


In [23]:
# To print memory requirements because the memory demands from host might be a lot
echo $MODEL
llama-fit-params -hf $MODEL -ub 64 -ngl 41 -ncmoe 36
#llama-fit-params -hf $MODEL -ub 64 -ngl 41 -ncmoe 40 --fit-print on

unsloth/Qwen3.6-35B-A3B-GGUF:UD-IQ4_NL
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
common_download_file_single_online: HEAD failed, status: 404
no remote preset found, skipping
common_params_fit_impl: getting device memory data for initial parameters:
common_memory_breakdown_print: | memory breakdown [MiB] | total   free     self   model   context   compute       unaccounted |
common_memory_breakdown_print: |   - CUDA0 (RTX 2060)   |  5737 = 5075 + ( 8942 =  3509 +    5182 +     250) + 17592186036135 |
common_memory_breakdown_print: |   - Host               |                 13750 = 13685 +       0 +      65                   |
common_params_fit_impl: projected to use 8942 MiB of device memory vs. 5075 MiB of free device memory
common_params_fit_impl: cannot me

In [83]:
echo "Staring llama-bench for ${MODEL}..." 
llama-bench -hf $MODEL -ub 128 -ot $(cat tensor.txt) --mmap 0 -ngl 99 -ncmoe 36 -p 1000

Staring llama-bench for unsloth/Qwen3.6-35B-A3B-GGUF:UD-IQ4_NL...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
error: unrecognized buffer type 'CPU"'
Available buffer types:
  CPU
  CUDA0
error: invalid parameter for argument: -p
usage: llama-bench [options]

options:
  -h, --help
  --numa <distribute|isolate|numactl>         numa mode (default: disabled)
  -r, --repetitions <n>                       number of times to repeat each test (default: 5)
  --prio <-1|0|1|2|3>                         process/thread priority (default: 0)
  --delay <0...N> (seconds)                   delay between each test (default: 0)
  -o, --output <csv|json|jsonl|md|sql>        output format printed to stdout (default: md)
  -oe, --output-err <csv|json|jsonl|md|sql>   output format p

: 1

In [59]:
echo "Staring llama-bench for ${MODEL}..." 
llama-bench -hf $MODEL -fitt 512 -ub 512,1024 --mmap 0 -ngl 99 -ncmoe 99 -p 1000

Staring llama-bench for unsloth/Qwen3.6-35B-A3B-GGUF:UD-IQ4_NL...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
| model                          |       size |     params | backend    | ngl |  n_cpu_moe | n_ubatch | mmap |       fitt |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | ---------: | -------: | ---: | ---------: | --------------: | -------------------: |
common_params_fit_impl: getting device memory data for initial parameters:
common_memory_breakdown_print: | memory breakdown [MiB] | total   free     self   model   context   compute       unaccounted |
common_memory_breakdown_print: |   - CUDA0 (RTX 2060)   |  5737 = 5158 + (17255 = 16679 +      82 +     493) + 17592186027739 |


In [ ]:
llama-fit-params -hf $MODEL -ncmoe 32

In [2]:
echo "Staring llama-cli for ${MODEL}..." 
llama-cli -lv 3 -hf $MODEL -ub 512 -fitt 512 -ngl -1 -ncmoe 99 --no-mmap --no-warmup --no-mmproj --reasoning off --single-turn --prompt "Who are you?"

Staring llama-cli for ...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
error: invalid argument: 512


: 1

In [5]:
echo "Staring llama-server for ${MODEL}..." 
cat args.txt | xargs llama-server -hf $MODEL --threads-http 1 --no-warmup --no-mmproj --reasoning off -np 1

Staring llama-server for unsloth/Qwen3.6-35B-A3B-GGUF:UD-IQ4_NL...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
common_download_file_single_online: HEAD failed, status: 404
no remote preset found, skipping
build_info: b8925-0adede866
system_info: n_threads = 4 (n_threads_batch = 4) / 4 | CUDA : ARCHS = 500,610,700,750,800,860,890,1200 | USE_GRAPHS = 1 | PEER_MAX_BATCH_SIZE = 128 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Running without SSL
init: using 1 threads for HTTP server
start: binding port with default address family
main: loading model
srv    load_model: loading model '/root/.cache/huggingface/hub/models--unsloth--Qwen3.6-35B-A3B-GGUF/snapshots/a483e9e6cbd595906af30beda3